# Hackathon: From Raw Data to ML-Ready Dataset
## Insight-Driven EDA and End-to-End Feature Engineering on Airbnb Data Using pandas and Plotly

### What is a Hackathon?

A hackathon is a fast-paced, collaborative event where participants use data and technology to solve a real problem end-to-end.  
In this hackathon, you will work with a **real-world Airbnb dataset** and complete two interconnected goals:

- Produce a **high-quality exploratory data analysis (EDA)** using `pandas` and `plotly`, extracting meaningful insights, trends, and signals from the data.  
- Design and deliver a **clean, feature-rich, ML-ready dataset** that will serve as the foundation for a follow-up hackathon focused on building and evaluating machine learning models.

Your task is to **get the most out of the data**: uncover structure and patterns through EDA, and engineer informative features (numerical, categorical, temporal, textual (TF–IDF), and optionally image-based) to maximize the predictive power of the final dataset.

<div class="alert alert-success">
<b>About the Dataset</b>

<u>Context</u>

The data comes from <a href="https://insideairbnb.com/get-the-data/">Inside Airbnb</a>, an open project that publishes detailed, regularly updated datasets for cities around the world.  
Each city provides three main CSV files:

- <b>listings.csv</b> — property characteristics, host profiles, descriptions, amenities, etc.  
- <b>calendar.csv</b> — daily availability and pricing information for each listing.  
- <b>reviews.csv</b> — guest feedback and textual reviews.

These datasets offer a rich view of the short-term rental market, including availability patterns, pricing behavior, host attributes, and guest sentiment.  

<u>Inspiration</u>

Your ultimate objective is to create a dataset suitable for training a machine learning model that predicts whether a specific Airbnb listing will be <b>available on a given date</b>, using property attributes, review information, and host characteristics.
</div>

<div class="alert alert-info">
<b>Task</b>

Using one city of your choice from Inside Airbnb, create an end-to-end pipeline that:

1. Loads and explores the raw data (EDA).  
2. Engineers features (numerical, categorical, temporal, textual TF–IDF, etc.).  
3. Builds a unified ML-ready dataset.  

Please remember to add comments explaining your decisions. Comments help us understand your thought process and ensure accurate evaluation of your work. This assignment requires code-based solutions—**manually calculated or hard-coded results will not be accepted**. Thoughtful comments and visualizations are encouraged and will be highly valued.

- Write your solution directly in this notebook, modifying it as needed.
- Once completed, submit the notebook in **.ipynb** format via Moodle.
    
<b>Collaboration Requirement: Git & GitHub</b>

You must collaborate with your team using a **shared GitHub repository**.  
Your use of Git is part of the evaluation. We will specifically look at:

- Commit quality (clear messages, meaningful steps).  
- Balanced participation across team members.  
- Use of branches.  
- Ability to resolve merge conflicts appropriately.  
- A clean, readable project history that reflects real collaboration.

Good Git practice is **part of your grade**, not optional.
</div>
<div class="alert alert-danger">
    You are free to add as many cells as you wish as long as you leave untouched the first one.
</div>

<div class="alert alert-warning">

<b>Hints</b>

- Text columns often carry substantial predictive power, use text-vectorization methods to extract meaningful features.  
- Make sure all columns use appropriate data types (categorical, numeric, datetime, boolean). Correct dtypes help prevent subtle bugs and improve performance.  
- Feel free to enrich the dataset with any additional information you consider useful: engineered features, external data, derived temporal features, etc.  
- If the dataset is too large for your computer, use <code>.sample()</code> to work with a subset while preserving the logic of your pipeline.  
- Plotly offers a wide variety of powerful visualizations, experiment creatively, but always begin with a clear analytical question: *What insight am I trying to uncover with this plot?*

</div>




<div class="alert alert-danger">
<b>Submission Deadline:</b> Wednesday, December 3rd, 12:00

Start with a simple, working pipeline.  
Do not over-complicate your code too much. Start with a simple working solution and refine it if you have time.
</div>

<div class="alert alert-danger">
    
You may add as many cells as you want, but the **first cell must remain exactly as provided**. Do not edit, move, or delete it under any circumstances.
</div>


In [7]:
# LEAVE BLANK

### Team Information

Fill in the information below.  
All fields are **mandatory**.

- **GitHub Repository URL**: Paste the link to the team repo you will use for collaboration.
- **Team Members**: List all student names (and emails or IDs if required).

Do not modify the section title.  
Do not remove this cell.


In [8]:
# === Team Information (Mandatory) ===
# Fill in the fields below.

GITHUB_REPO = "https://github.com/ayushr-1o/hackathonx.git"       # e.g. "https://github.com/myteam/airbnb-hackathon"
TEAM_MEMBERS = [
    # "Ayush Raj",
    # "Lucas Haesaert",
    # "Fabrizio Icauzio",
    # "Lara Isikci"
    # "Eng Pongtangya"

]

GITHUB_REPO, TEAM_MEMBERS




('https://github.com/ayushr-1o/hackathonx.git', [])

In [9]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import plotly.express as px
import plotly.graph_objects as go



# Go up one level (to Desktop) and find the file
df_listings = pd.read_csv('https://data.insideairbnb.com/the-netherlands/north-holland/amsterdam/2025-09-11/data/listings.csv.gz')
df_calendar = pd.read_csv('https://data.insideairbnb.com/the-netherlands/north-holland/amsterdam/2025-09-11/data/calendar.csv.gz')
df_reviews = pd.read_csv('https://data.insideairbnb.com/the-netherlands/north-holland/amsterdam/2025-09-11/data/reviews.csv.gz')

# Down-sample calendar to keep the notebook responsive while respecting the
# requirement of using at least 10% of the rows.
CALENDAR_SAMPLE_FRAC = 0.10
if 0 < CALENDAR_SAMPLE_FRAC < 1:
    df_calendar = (
        df_calendar
        .sample(frac=CALENDAR_SAMPLE_FRAC, random_state=42)
        .sort_values(['listing_id', 'date'])
        .reset_index(drop=True)
    )
print('Calendar sample size:', len(df_calendar))

Calendar sample size: 382520


In [10]:
from datetime import datetime

# --- Helper functions -------------------------------------------------------
def clean_currency(series: pd.Series) -> pd.Series:
    """Remove currency symbols/commas and cast to float."""
    cleaned = series.astype(str).str.replace(r"[$,]", "", regex=True).str.strip()
    return pd.to_numeric(cleaned.replace({'': np.nan, 'nan': np.nan}), errors='coerce')

def clean_percentage(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.replace('%', '').str.strip()
    return pd.to_numeric(cleaned.replace({'': np.nan, 'nan': np.nan}), errors='coerce')

def tf_to_int(series: pd.Series) -> pd.Series:
    mapper = {'t': 1, 'f': 0, 'True': 1, 'False': 0, True: 1, False: 0}
    return series.map(mapper).astype('Int64')

# Establish a snapshot date (latest date in the calendar) for tenure/recency feats
calendar_max_date = pd.to_datetime(df_calendar['date']).max()

# --- Listings cleaning & feature engineering --------------------------------
listing_cols = [
    'id', 'description', 'neighborhood_overview', 'amenities',
    'neighbourhood_cleansed', 'latitude', 'longitude',
    'property_type', 'room_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms_text',
    'host_id', 'host_since', 'host_response_time', 'host_response_rate',
    'host_is_superhost', 'host_identity_verified', 'host_acceptance_rate',
    'host_listings_count', 'host_total_listings_count',
    'number_of_reviews', 'review_scores_rating', 'reviews_per_month',
    'price', 'minimum_nights', 'maximum_nights', 'instant_bookable'
]
existing_cols = [col for col in listing_cols if col in df_listings.columns]
df_listings_clean = df_listings[existing_cols].copy()

if 'price' in df_listings_clean.columns:
    df_listings_clean['base_listing_price'] = clean_currency(df_listings_clean.pop('price'))
if 'host_response_rate' in df_listings_clean.columns:
    df_listings_clean['host_response_rate'] = clean_percentage(df_listings_clean['host_response_rate'])
if 'host_acceptance_rate' in df_listings_clean.columns:
    df_listings_clean['host_acceptance_rate'] = clean_percentage(df_listings_clean['host_acceptance_rate'])
for bool_col in ['instant_bookable', 'host_is_superhost', 'host_identity_verified']:
    if bool_col in df_listings_clean.columns:
        df_listings_clean[bool_col] = tf_to_int(df_listings_clean[bool_col])

if 'host_since' in df_listings_clean.columns:
    df_listings_clean['host_since'] = pd.to_datetime(df_listings_clean['host_since'], errors='coerce')
    df_listings_clean['host_tenure_days'] = (calendar_max_date - df_listings_clean['host_since']).dt.days

if 'bathrooms_text' in df_listings_clean.columns:
    numeric_bath = df_listings_clean['bathrooms_text'].str.extract(r'([0-9]*\.?[0-9]+)')
    df_listings_clean['bathrooms'] = pd.to_numeric(numeric_bath[0], errors='coerce')
    df_listings_clean['shared_bathroom_flag'] = df_listings_clean['bathrooms_text'].str.contains('shared', case=False, na=False)

if 'amenities' in df_listings_clean.columns:
    df_listings_clean['amenities_count'] = df_listings_clean['amenities'].fillna('[]').apply(lambda x: x.count(',') + 1 if x != '[]' else 0)

text_features = ['description', 'neighborhood_overview']
for col in text_features:
    if col in df_listings_clean.columns:
        df_listings_clean[f'{col}_word_count'] = df_listings_clean[col].fillna('').str.split().str.len()
        df_listings_clean[f'{col}_char_count'] = df_listings_clean[col].fillna('').str.len()

# Vectorize listing descriptions (bag-of-words signal for availability modeling)
if 'description' in df_listings_clean.columns:
    desc_corpus = df_listings_clean['description'].fillna('').astype(str)
    if desc_corpus.str.strip().str.len().sum() > 0:
        desc_vectorizer = TfidfVectorizer(
            max_features=150,
            ngram_range=(1, 2),
            min_df=15,
            stop_words='english'
        )
        desc_matrix = desc_vectorizer.fit_transform(desc_corpus)
        desc_cols = [f"desc_tfidf_{feat.replace(' ', '_')[:40]}" for feat in desc_vectorizer.get_feature_names_out()]
        df_desc_tfidf = pd.DataFrame(desc_matrix.toarray(), columns=desc_cols, index=df_listings_clean.index)
        df_listings_clean = pd.concat([df_listings_clean, df_desc_tfidf], axis=1)

numeric_cols = ['accommodates', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights',
                'host_listings_count', 'host_total_listings_count', 'number_of_reviews',
                'review_scores_rating', 'reviews_per_month']
for col in numeric_cols:
    if col in df_listings_clean.columns:
        df_listings_clean[col] = pd.to_numeric(df_listings_clean[col], errors='coerce')

print('Listings prepared with enriched features:', df_listings_clean.shape)

# --- Reviews aggregation ----------------------------------------------------
df_reviews_clean = df_reviews.copy()
df_reviews_clean['date'] = pd.to_datetime(df_reviews_clean['date'], errors='coerce')
df_reviews_clean['comment_length'] = df_reviews_clean['comments'].fillna('').str.len()
df_reviews_clean['has_comment'] = df_reviews_clean['comments'].notna().astype(int)

review_groupers = {
    'total_reviews': ('id', 'count'),
    'unique_reviewers': ('reviewer_id', 'nunique'),
    'avg_comment_length': ('comment_length', 'mean'),
    'pct_reviews_with_text': ('has_comment', 'mean'),
    'first_review_date': ('date', 'min'),
    'last_review_date': ('date', 'max')
}

df_reviews_agg = (
    df_reviews_clean
    .groupby('listing_id')
    .agg(**review_groupers)
    .reset_index()
)

df_reviews_agg['review_active_days'] = (df_reviews_agg['last_review_date'] - df_reviews_agg['first_review_date']).dt.days
recency_reference = calendar_max_date if pd.notna(calendar_max_date) else pd.to_datetime('today')
df_reviews_agg['review_recency_days'] = (recency_reference - df_reviews_agg['last_review_date']).dt.days
active_months = np.maximum(df_reviews_agg['review_active_days'] / 30.44, 1)
df_reviews_agg['reviews_per_active_month'] = df_reviews_agg['total_reviews'] / active_months

# Vectorize concatenated review comments per listing
df_review_comment_tfidf = pd.DataFrame()
review_corpus = (
    df_reviews_clean
    .dropna(subset=['listing_id'])
    .groupby('listing_id')['comments']
    .apply(lambda x: ' '.join(x.dropna().astype(str)))
    .reset_index(name='review_comment_corpus')
)
if not review_corpus.empty:
    review_corpus['review_comment_corpus'] = review_corpus['review_comment_corpus'].fillna('')
    review_vectorizer = TfidfVectorizer(
        max_features=150,
        ngram_range=(1, 2),
        min_df=30,
        stop_words='english'
    )
    review_matrix = review_vectorizer.fit_transform(review_corpus['review_comment_corpus'])
    review_cols = [f"review_tfidf_{feat.replace(' ', '_')[:40]}" for feat in review_vectorizer.get_feature_names_out()]
    df_review_comment_tfidf = pd.concat(
        [review_corpus[['listing_id']].reset_index(drop=True),
         pd.DataFrame(review_matrix.toarray(), columns=review_cols)],
        axis=1
    )
    df_reviews_agg = df_reviews_agg.merge(df_review_comment_tfidf, on='listing_id', how='left')
    print('Review comment TF-IDF features added:', len(review_cols))
else:
    print('Review comment TF-IDF skipped: no comments available')

print('Review aggregates ready (with TF-IDF):', df_reviews_agg.shape)

# --- Calendar cleaning ------------------------------------------------------
df_calendar_clean = df_calendar.copy()
df_calendar_clean['date'] = pd.to_datetime(df_calendar_clean['date'], errors='coerce')
for col in ['price', 'adjusted_price']:
    if col in df_calendar_clean.columns:
        df_calendar_clean[col] = clean_currency(df_calendar_clean[col])
        df_calendar_clean = df_calendar_clean.rename(columns={col: f'calendar_{col}'})
for col in ['minimum_nights', 'maximum_nights']:
    if col in df_calendar_clean.columns:
        df_calendar_clean[col] = pd.to_numeric(df_calendar_clean[col], errors='coerce')
        df_calendar_clean = df_calendar_clean.rename(columns={col: f'calendar_{col}'})
if 'available' in df_calendar_clean.columns:
    df_calendar_clean['available_flag'] = tf_to_int(df_calendar_clean['available']).astype('Int8')

print('Calendar cleaned:', df_calendar_clean.shape)

Listings prepared with enriched features: (10480, 187)
Review comment TF-IDF features added: 150
Review aggregates ready (with TF-IDF): (9383, 160)


C:\Users\fabri\AppData\Local\Temp\ipykernel_42548\2441038404.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return pd.to_numeric(cleaned.replace({'': np.nan, 'nan': np.nan}), errors='coerce')


Calendar cleaned: (382520, 8)


C:\Users\fabri\AppData\Local\Temp\ipykernel_42548\2441038404.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return pd.to_numeric(cleaned.replace({'': np.nan, 'nan': np.nan}), errors='coerce')


In [11]:
# Merge listings (with engineered features) + review aggregates

df_properties = (
    df_listings_clean
    .merge(df_reviews_agg, left_on='id', right_on='listing_id', how='left')
    .drop(columns=['listing_id'], errors='ignore')
)

# Merge onto the calendar spine to obtain the ML-ready dataset

df_final = (
    df_calendar_clean
    .merge(df_properties, left_on='listing_id', right_on='id', how='left', validate='many_to_one')
    .drop(columns=['id'], errors='ignore')
    .sort_values(['listing_id', 'date'])
    .reset_index(drop=True)
)

print('Final dataset ready! Rows:', len(df_final), '| Columns:', df_final.shape[1])
print('Feature sample:', df_final.columns[:15].tolist(), '...')
df_final.head()

Final dataset ready! Rows: 382520 | Columns: 353
Feature sample: ['listing_id', 'date', 'available', 'calendar_price', 'calendar_adjusted_price', 'calendar_minimum_nights', 'calendar_maximum_nights', 'available_flag', 'description', 'neighborhood_overview', 'amenities', 'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type'] ...


,listing_id,date,available,calendar_price,calendar_adjusted_price,calendar_minimum_nights,calendar_maximum_nights,available_flag,description,neighborhood_overview,...,review_tfidf_und,review_tfidf_visit,review_tfidf_walk,review_tfidf_walking,review_tfidf_walking_distance,review_tfidf_war,review_tfidf_welcoming,review_tfidf_wir,review_tfidf_wonderful,review_tfidf_zu
0,27886,2025-09-11,f,NaN,NaN,3,30,0,Stylish and romantic houseboat on fantastic hi...,"Central, quiet, safe, clean and beautiful.",...,0.095988,0.034163,0.060645,0.037925,0.030007,0.032048,0.025058,0.043479,0.097564,0.041569
1,27886,2025-09-18,f,NaN,NaN,3,30,0,Stylish and romantic houseboat on fantastic hi...,"Central, quiet, safe, clean and beautiful.",...,0.095988,0.034163,0.060645,0.037925,0.030007,0.032048,0.025058,0.043479,0.097564,0.041569
2,27886,2025-09-21,f,NaN,NaN,3,30,0,Stylish and romantic houseboat on fantastic hi...,"Central, quiet, safe, clean and beautiful.",...,0.095988,0.034163,0.060645,0.037925,0.030007,0.032048,0.025058,0.043479,0.097564,0.041569
3,27886,2025-09-26,f,NaN,NaN,3,30,0,Stylish and romantic houseboat on fantastic hi...,"Central, quiet, safe, clean and beautiful.",...,0.095988,0.034163,0.060645,0.037925,0.030007,0.032048,0.025058,0.043479,0.097564,0.041569
4,27886,2025-10-19,f,NaN,NaN,3,30,0,Stylish and romantic houseboat on fantastic hi...,"Central, quiet, safe, clean and beautiful.",...,0.095988,0.034163,0.060645,0.037925,0.030007,0.032048,0.025058,0.043479,0.097564,0.041569


In [14]:
# --- Plotly visualizations ---------------------------------------------------

# 1) Daily availability trend (helps gauge seasonal supply)
daily_availability = (
    df_final
    .groupby('date', as_index=False)['available_flag']
    .mean()
    .rename(columns={'available_flag': 'avg_availability'})
)
fig_daily = px.line(
    daily_availability,
    x='date',
    y='avg_availability',
    title='Daily Availability Rate (Sampled Calendar)',
    labels={'avg_availability': 'Avg availability probability'},
)
fig_daily.update_traces(mode='lines+markers', marker=dict(size=4))
fig_daily.show()

# 2) Price vs. capacity scatter colored by room type
price_capacity = df_properties[['room_type', 'accommodates', 'base_listing_price']].dropna()
fig_price_capacity = px.scatter(
    price_capacity,
    x='accommodates',
    y='base_listing_price',
    color='room_type',
    title='Base Price vs. Capacity by Room Type',
    labels={'accommodates': 'Guests accommodated', 'base_listing_price': 'Base price (EUR)'}
)
fig_price_capacity.update_traces(marker=dict(size=6, opacity=0.65))
fig_price_capacity.show()

# 3) Geospatial distribution of listings by price (sampled for readability)
geo_sample = (
    df_properties[['latitude', 'longitude', 'base_listing_price', 'room_type']]
    .dropna()
    .sample(n=min(2500, len(df_properties)), random_state=42)
)
geo_sample = geo_sample[geo_sample['base_listing_price'] > 0]
geo_sample['log_price'] = np.log10(geo_sample['base_listing_price'])

fig_geo = px.scatter_map(
    geo_sample,
    lat='latitude',
    lon='longitude',
    color='log_price',
    hover_data={
        'room_type': True,
        'base_listing_price': True,
        'log_price': False
    },
    color_continuous_scale='Viridis',
    title='Listing Price Hotspots (log10 scale, Sample)',
    zoom=10,
    height=600,
)
fig_geo.update_traces(marker=dict(size=7, opacity=0.8))
fig_geo.update_layout(
    map_style='carto-positron',
    coloraxis_colorbar=dict(title='log10 price')
)
fig_geo.show()

# 4) Host responsiveness vs. review quality
host_review = df_properties[['host_response_rate', 'review_scores_rating', 'reviews_per_month']].dropna()
fig_host_review = px.scatter(
    host_review,
    x='host_response_rate',
    y='review_scores_rating',
    size='reviews_per_month',
    title='Host Response Rate vs. Review Score',
    labels={'host_response_rate': 'Host response rate (%)', 'review_scores_rating': 'Review score rating'},
    color='reviews_per_month',
    color_continuous_scale='Plasma'
)
fig_host_review.update_traces(marker=dict(sizemode='diameter', opacity=0.7))
fig_host_review.show()


# 6) Availability heatmap (month vs weekday)
availability_calendar = df_final[['date', 'available_flag']].dropna()
if not availability_calendar.empty:
    availability_calendar = availability_calendar.assign(
        month=lambda d: d['date'].dt.month_name(),
        weekday=lambda d: d['date'].dt.day_name()
    )
    availability_matrix = (
        availability_calendar
        .groupby(['month', 'weekday'])['available_flag']
        .mean()
        .reset_index()
    )
    months_order = ['January','February','March','April','May','June','July','August','September','October','November','December']
    weekdays_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    availability_matrix['month'] = pd.Categorical(availability_matrix['month'], categories=months_order, ordered=True)
    availability_matrix['weekday'] = pd.Categorical(availability_matrix['weekday'], categories=weekdays_order, ordered=True)
    availability_pivot = availability_matrix.pivot(index='weekday', columns='month', values='available_flag')
    fig_availability_heatmap = px.imshow(
        availability_pivot,
        color_continuous_scale='Viridis',
        aspect='auto',
        title='Availability Rate by Weekday and Month'
    )
    fig_availability_heatmap.update_coloraxes(colorbar_title='Availability rate')
    fig_availability_heatmap.show()